# Japanese model mini

In [9]:
##Removing annoying warnings
import warnings
import os

#IProgress
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*IProgress not found.*"
)

#Suppress the torch/cuda AMD SMI warning
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    message=".*Can't initialize amdsmi.*"
)

warnings.filterwarnings("ignore")

import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=UserWarning, message=".*IProgress not found.*")
warnings.filterwarnings("ignore", category=UserWarning, message=".*Can't initialize amdsmi.*")
warnings.filterwarnings("ignore")
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"

In [10]:
#Tokenizer
tokenizer = AutoTokenizer.from_pretrained("rinna/japanese-gpt-neox-3.6b-instruction-sft-v2", use_fast=False)

#Config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

#Base Model - NO FINETUNING
if torch.cuda.is_available():
    model = AutoModelForCausalLM.from_pretrained(
        "rinna/japanese-gpt-neox-3.6b-instruction-sft-v2",
        quantization_config=bnb_config,
        device_map="auto"
    )
    print("CUDA is available! GPU is ready to use.")
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("CUDA is not available.")

Loading weights: 100%|██████████| 436/436 [00:11<00:00, 38.56it/s, Materializing param=gpt_neox.layers.35.post_attention_layernorm.weight]  


CUDA is available! GPU is ready to use.
GPU Device: AMD Radeon RX 7800 XT


In [11]:
#dataset
ds = load_dataset("NilanE/ParallelFiction-Ja_En-100k")
split = ds['train'].train_test_split(test_size=0.01)
ds = split["test"].select(range(200))

print(f"Loaded {len(ds)} examples for generation.")

Loaded 200 examples for generation.


In [12]:
results = []
total_generations = len(ds)

print("Starting generation...")

for i, row in enumerate(ds):
    # --- CORRECTED LINE: Access nested genres ---
    genres_list = row['meta']['novelupdates']['genres']
    genres_str = ", ".join(genres_list)
    
    # --- PROMPT CONSTRUCTION ---
    # Matches original notebook: "「Genre, Genre」のジャンルが含まれるストーリーを書いて。"
    instruction = f"「{genres_str}」のジャンルが含まれるストーリーを書いて。"
    input_text = f"ユーザー: {instruction}<NL>システム: "
    # ---------------------------
    
    # Tokenize
    token_ids = tokenizer.encode(input_text, add_special_tokens=False, return_tensors="pt")
    
    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            token_ids.to(model.device),
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    # Decode
    full_text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
    
    # Post-process: Remove prompt
    story_content = full_text.replace(input_text.replace("<NL>", "\n"), "").strip()
    
    results.append({
        "genres": genres_str,
        "story": story_content
    })
    
    # Print progress every 10 stories to reduce clutter
    if (i + 1) % 10 == 0:
        print(f"Generated story {i+1}/{total_generations}")

# Save to generated_before.csv
df = pd.DataFrame(results)
df.to_csv("generated_before.csv", index=False, encoding="utf-8-sig")
print("--- Success! Stories saved to generated_before.csv ---")

Starting generation...
Generated story 10/200
Generated story 20/200
Generated story 30/200
Generated story 40/200
Generated story 50/200
Generated story 60/200
Generated story 70/200
Generated story 80/200
Generated story 90/200
Generated story 100/200
Generated story 110/200
Generated story 120/200
Generated story 130/200
Generated story 140/200
Generated story 150/200
Generated story 160/200
Generated story 170/200
Generated story 180/200
Generated story 190/200
Generated story 200/200
--- Success! Stories saved to generated_before.csv ---
